In poctest.ipynb werden de modellen getest. Nu word hierop verder gebouwd om tot een resultaat te bekomen waardat bij een foto van een winkelschap alle producten herkent worden en correct beschreven worden.

De structuur die we volgen voor de pipeline op te stellen is als volgt:
- Image
-  ↓
- YOLO herkent waar producten staan
-  ↓
- Crop de producten die gevonden worden
-  ↓
- CLIP + Florence-2 maken weight voor elke crop
-  ↓
- FAISS zoekt naar afbeelding met similar weight in de openfoodfacts database zodat het zelfde product gevonden kan worden

Imports

In [1]:
from ultralytics import YOLO, RTDETR
from transformers import Owlv2Processor, Owlv2ForObjectDetection, MobileViTImageProcessor, MobileViTModel, AutoProcessor, AutoModelForCausalLM, CLIPProcessor, CLIPModel
import torch
from nanoowl.owl_predictor import OwlPredictor
from PIL import Image
import cv2
import sys
import types
import importlib.machinery
import os
import clip
import pandas as pd
import duckdb
import numpy as np
import faiss
import uuid
from datetime import datetime
import requests
import polars as pl
from io import BytesIO

device = "cuda" if torch.cuda.is_available() else "cpu"

c:\Users\krist\Documents\Hogent_IT\3de_jaar\Bachelorproef\bachproef26\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Foodfacts database met urls al in

In [2]:
df_foodfacts = duckdb.query("""
    SELECT *
    FROM 'C:\\Users\\krist\\Documents\\Hogent_IT\\3de_jaar\\Bachelorproef\\bachproef26\\latex-hogent-bachproef\\bachproef\\datasets\\food_images_long.parquet'
""").pl()

print(df_foodfacts.columns)
print(len(df_foodfacts))

['code', 'product_name', 'image_key', 'image_url']
441913


In [ ]:
# def build_image_url(code, key, rev, size="400"):
#     code = str(code).zfill(13)

#     prefix = code[:-4]
#     suffix = code[-4:]

#     parts = [prefix[i:i+3] for i in range(0, len(prefix), 3) if prefix[i:i+3]]
#     parts.append(suffix)

#     path = "/".join(parts)
#     url = f"https://images.openfoodfacts.org/images/products/{path}/{key}.{rev}.{size}.jpg"
#     print(url)
#     return url

# def extract_front_image(row):
#     images = row["images"]

#     if images is None:
#         return None

#     priority = ["front_nl", "front_fr", "front_en", "front"]

#     for pref in priority:
#         for img in images:
#             key = img.get("key")
#             rev = img.get("rev")

#             if key and rev and key.startswith(pref):
#                 return build_image_url(row["code"], key, rev)

#     return None

Florence-2 model

In [16]:
flash_attn = types.ModuleType("flash_attn")
flash_attn.__spec__ = importlib.machinery.ModuleSpec("flash_attn", None)

flash_attn_interface = types.ModuleType("flash_attn.flash_attn_interface")
flash_attn_interface.__spec__ = importlib.machinery.ModuleSpec("flash_attn.flash_attn_interface", None)

sys.modules["flash_attn"] = flash_attn
sys.modules["flash_attn.flash_attn_interface"] = flash_attn_interface

os.environ["FLASH_ATTENTION_FORCE_DISABLED"] = "1"

fl2_model_id = "microsoft/Florence-2-base"

fl2_processor = AutoProcessor.from_pretrained(fl2_model_id, trust_remote_code=True)

fl2_model = AutoModelForCausalLM.from_pretrained(
    fl2_model_id,
    trust_remote_code=True,
    attn_implementation="eager"
)

fl2_model.eval()

Florence2ForConditionalGeneration(
  (vision_tower): DaViT(
    (convs): ModuleList(
      (0): ConvEmbed(
        (proj): Conv2d(3, 128, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
        (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      )
      (1): ConvEmbed(
        (proj): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      )
      (2): ConvEmbed(
        (proj): Conv2d(256, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      )
      (3): ConvEmbed(
        (proj): Conv2d(512, 1024, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      )
    )
    (blocks): ModuleList(
      (0): MySequential(
        (0): MySequential(
          (spatial_block): SpatialBlock(
            (conv1): PreNorm(
              (fn): Depth

CLIP model

In [ ]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

clip_model.eval()

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

YOLO loslaten op de winkelschapafbeeldingen

In [11]:
def load_image(url):
    r = requests.get(url, timeout=10)
    return Image.open(BytesIO(r.content)).convert("RGB")

def get_clip_embedding(url):
    image = load_image(url)
    inputs = clip_processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        feat = clip_model.get_image_features(**inputs)

    feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.squeeze(0) 

In [17]:
def get_florence_caption(url):
    image = load_image(url)

    task = "<DETAILED_CAPTION>"
    inputs = fl2_processor(text=task, images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = fl2_model.generate(**inputs, max_new_tokens=100)

    caption = fl2_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return caption

In [ ]:
product_embeddings = {}

for code, group in df_foodfacts.group_by("code"):
    vectors = []

    for url in group["image_url"].unique():
        try:
            vec = get_clip_embedding(url)
            vectors.append(vec)
        except:
            continue

    if len(vectors) == 0:
        continue

    product_vec = torch.stack(vectors).mean(dim=0)
    product_vec = product_vec / product_vec.norm()

    product_embeddings[code] = product_vec.cpu()

In [ ]:
def get_text_embedding(text):
    inputs = clip_processor(text=[text], return_tensors="pt").to(device)

    with torch.no_grad():
        text_feat = clip_model.get_text_features(**inputs)

    text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
    return text_feat.squeeze(0)

In [ ]:
def get_fused_embedding(url):
    image = load_image(url)

    # CLIP image embedding
    img_inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        img_feat = clip_model.get_image_features(**img_inputs)

    img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

    # Florence caption
    caption = get_florence_caption(url)

    # CLIP text embedding
    text_feat = get_text_embedding(caption)

    # fusion (weighted)
    fused = 0.7 * img_feat + 0.3 * text_feat
    fused = fused / fused.norm()

    return fused.squeeze(0)

In [ ]:
product_embeddings = {}

for code, group in df_foodfacts.group_by("code"):
    vectors = []

    for url in group["image_url"].unique():
        try:
            vec = get_fused_embedding(url)
            vectors.append(vec)
        except:
            continue

    if not vectors:
        continue

    product_vec = torch.stack(vectors).mean(dim=0)
    product_vec = product_vec / product_vec.norm()

    product_embeddings[code] = product_vec.cpu()

In [ ]:
rows = []

for code, vec in product_embeddings.items():
    rows.append({
        "code": code,
        "embedding": vec.numpy() 
    })

df_embeddings = pd.DataFrame(rows)

In [ ]:
df_embeddings["embedding"] = df_embeddings["embedding"].apply(lambda x: x.tolist())

df_embeddings.to_parquet("product_embeddings.parquet", index=False)

In [ ]:
df_embeddings = pd.read_parquet("product_embeddings.parquet")


product_embeddings = {
    row["code"]: torch.tensor(row["embedding"])
    for _, row in df_embeddings.iterrows()
}

In [ ]:
import torch.nn.functional as F

def compare(code1, code2):
    v1 = product_embeddings[code1]
    v2 = product_embeddings[code2]

    return F.cosine_similarity(v1, v2, dim=0).item()

In [ ]:
similarity = compare("00011754", "4056489542568")
print(similarity)